# ShoeCo: multiperiod production planning with backlog

This notebook extends [08-ShoeCo.ipynb](08-ShoeCo.ipynb) by allowing late deliveries. It contains all its own data and can be run independently.

ShoeCo must plan production for the next four months. It begins with 500 pairs of shoes in inventory and 100 workers. Forecast demand is:

| Month | 1 | 2 | 3 | 4 |
| --- | --- | --- | --- | --- |
| Demand (pairs) | 3000 | 5000 | 2000 | 1000 |

Each worker is paid 1500 dollars per month and provides up to 160 regular working hours. Each worker can also work up to 20 overtime hours per month, paid at 13 dollars per hour. Hiring a worker costs 1600 dollars; firing a worker costs 2000 dollars. Each pair of shoes requires four labor hours and 15 dollars of raw materials. Holding inventory costs three dollars per pair remaining at the end of each month.

Demand may be postponed, with a penalty of 20 dollars per pair still **backlogged at the end of each month**. A pair delayed for two month ends incurs the penalty twice. All demand must be filled by the end of month 4. Choose production, staffing, overtime, inventory, and backlog to minimize total cost.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. JuMP, HiGHS, DataFrames, and Printf are already included. No external data files are needed.

## Problem data and timing

Hiring and firing occur at the start of a month, so the resulting workforce earns that month's salary and supplies that month's labor. Production is available to fill orders during that same month. Inventory is measured after accounting for demand at month end.

All decisions are continuous in this linear program, including the numbers of workers hired, employed, and fired. We will discuss integer decisions later in the course. There is no required final workforce, no severance charge after the planning horizon, and no value assigned to leftover shoes after month 4. Holding costs still apply to final inventory.

Named parameters collect the numerical inputs in one place. `d` is a one-dimensional demand vector, and `T` sets the number of months. The overtime rate is **13 dollars/hour**, matching the problem statement.

In [ ]:
using JuMP, HiGHS, DataFrames, Printf
import MathOptInterface as MOI

d = [3000, 5000, 2000, 1000]  # Demand in pairs of shoes.
T = length(d)
months = 1:T

initial_inventory = 500
initial_workforce = 100
regular_hours_per_worker = 160
overtime_hours_per_worker = 20
labor_hours_per_pair = 4

material_cost = 15       # Dollars per pair produced.
wage_cost = 1500         # Dollars per worker per month.
overtime_cost = 13      # Dollars per overtime hour.
hiring_cost = 1600       # Dollars per worker hired.
firing_cost = 2000       # Dollars per worker fired.
holding_cost = 3         # Dollars per pair at each month end.
backlog_cost = 20       # Dollars per pair backlogged at each month end.

## Decision variables

For each month $t = 1,\ldots,T$, use:

| Variable | Meaning | Units |
| --- | --- | --- |
| $x_t$ | Shoes produced | Pairs |
| $w_t$ | Workers employed after hiring and firing | Workers |
| $o_t$ | Overtime used | Hours |
| $h_t$ | Workers hired at the start of the month | Workers |
| $f_t$ | Workers fired at the start of the month | Workers |
| $i_t$ | Net inventory at month end | Pairs; may be negative |
| $L_t$ | Physical inventory left at month end | Pairs |
| $S_t$ | Unfilled demand at month end | Pairs |

All variables except $i_t$ are nonnegative. Net inventory is a signed quantity: a negative value means that some demand remains unfilled. Separate nonnegative variables let us charge different rates for physical inventory and backlog:

$$
i_t = L_t - S_t.
$$

With positive holding and backlog costs, an optimum cannot have both $L_t > 0$ and $S_t > 0$ in the same month. Reducing both by the same amount would preserve net inventory and reduce cost. Thus, at an optimum, $L_t = \max(i_t,0)$ and $S_t = \max(-i_t,0)$, without adding a nonlinear constraint.

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model)  # Remove this line to see the solver log.

@variable(model, x[1:T] >= 0)
@variable(model, w[1:T] >= 0)
@variable(model, o[1:T] >= 0)
@variable(model, h[1:T] >= 0)
@variable(model, f[1:T] >= 0)
@variable(model, i[1:T])  # Net inventory can be negative.
@variable(model, L[1:T] >= 0)
@variable(model, S[1:T] >= 0)

@constraint(model, inventory_identity[t in months], i[t] == L[t] - S[t])

## Minimize total cost

The objective includes materials, regular wages, overtime, hiring, firing, and inventory holding costs, plus backlog penalties:

$$
\min \sum_{t=1}^{T}\left(
15x_t + 1500w_t + 13o_t + 1600h_t + 2000f_t + 3L_t + 20S_t
\right).
$$

Regular wages are paid for every employed worker, even when some regular hours are unused. Overtime is an additional hourly expense. Every term in the objective is measured in dollars. Named JuMP expressions let us report each cost component after solving.

In [ ]:
@expression(model, material_expense, material_cost * sum(x))
@expression(model, wage_expense, wage_cost * sum(w))
@expression(model, overtime_expense, overtime_cost * sum(o))
@expression(model, hiring_expense, hiring_cost * sum(h))
@expression(model, firing_expense, firing_cost * sum(f))
@expression(model, holding_expense, holding_cost * sum(L))
@expression(model, backlog_expense, backlog_cost * sum(S))

@objective(model, Min,
    material_expense + wage_expense + overtime_expense +
    hiring_expense + firing_expense + holding_expense + backlog_expense)

## Labor capacity and balances across months

Production cannot use more labor than the workforce and overtime provide:

$$
4x_t \leq 160w_t + o_t, \qquad o_t \leq 20w_t
\quad \forall t.
$$

The first inequality allows unused regular hours. The second links the overtime limit to the workforce employed that month.

Inventory and workforce carry information from one month to the next. With initial values $i_0 = 500$ and $w_0 = 100$,

$$
i_{t-1} + x_t = d_t + i_t, \qquad
w_{t-1} + h_t - f_t = w_t
\quad \forall t.
$$

We write the first-month balances separately to insert the initial inventory and workforce. The remaining months use the previous month's decision variables.

Here $d_t$ is demand arriving in month $t$, which need not equal deliveries that month. The inventory equation tracks **net** inventory, so unmet demand carries forward automatically. For example, $i_{t-1} = -100$ means production must cover 100 previously backlogged pairs in addition to the new demand before any physical inventory can remain.

We explicitly impose $S_T = 0$ to clear all backlog by the end of the horizon. Together with $i_T = L_T - S_T$, this also gives $i_T \geq 0$. Omitting the terminal requirement would allow some orders to remain unfilled after month 4.

In [ ]:
@constraint(model, production[t in months],
    labor_hours_per_pair * x[t] <= regular_hours_per_worker * w[t] + o[t])
@constraint(model, overtime[t in months],
    o[t] <= overtime_hours_per_worker * w[t])

@constraint(model, inv_bal_init, initial_inventory + x[1] == d[1] + i[1])
@constraint(model, inv_bal[t in 2:T], i[t - 1] + x[t] == d[t] + i[t])

@constraint(model, work_bal_init, initial_workforce + h[1] - f[1] == w[1])
@constraint(model, work_bal[t in 2:T], w[t - 1] + h[t] - f[t] == w[t])

@constraint(model, clear_backlog, S[T] == 0)

model

## Solve and inspect the monthly plan

Check that HiGHS found an optimum and that a feasible solution is available before reading values. The table includes hiring and firing as well as production and staffing, so we can follow both balances month by month.

Keep the unrounded values when checking feasibility. The LP permits fractional workers; rounding decisions independently can break the workforce balances or leave too little production capacity. An implementable plan with whole workers calls for integer restrictions and a new solve, which we will study later.

In [ ]:
optimize!(model)
status = termination_status(model)
println("Termination status: ", status)
status == MOI.OPTIMAL || error("HiGHS stopped with status $(status).")
is_solved_and_feasible(model) || error("No feasible optimal solution is available.")

minimum_cost = objective_value(model)
@printf("\nMinimum total cost: \$%.2f\n", minimum_cost)

plan = DataFrame(
    month = collect(months),
    demand = d,
    produced = value.(x),
    workers = value.(w),
    hired = value.(h),
    fired = value.(f),
    overtime_hours = value.(o),
    net_inventory = value.(i),
    inventory = value.(L),
    backlog = value.(S),
)
plan

## Account for the costs

Each row sums a cost over all months. Compare these components with the monthly plan: salaries are paid even when a worker has idle time, whereas overtime is charged only for the hours used. Holding costs are charged on each month's ending physical inventory.

Backlog penalties likewise apply to each month's ending unfilled demand; they are not a one-time charge per delayed order.

In [ ]:
cost_report = DataFrame(
    component = ["Materials", "Regular wages", "Overtime", "Hiring", "Firing", "Inventory holding", "Backlog penalties"],
    dollars = [
        value(material_expense),
        value(wage_expense),
        value(overtime_expense),
        value(hiring_expense),
        value(firing_expense),
        value(holding_expense),
        value(backlog_expense),
    ],
)
cost_report

## Interpret the backlog decision

With the supplied data, the minimum total cost is **690,000 dollars**, a saving of **2500 dollars** compared with the on-time plan in [08-ShoeCo.ipynb](08-ShoeCo.ipynb). Allowing backlog cannot increase the minimum cost: every feasible on-time plan remains feasible here with $S_t = 0$ and $L_t = i_t$.

Which months have backlogged orders, and when are those orders filled? Does the saving come from lower salaries, less overtime, fewer workforce changes, or a combination? Why is a backlog penalty sometimes worth paying?

At the final month, verify that backlog is zero and that initial inventory plus total production equals total demand plus final inventory. For these data, there is no final inventory, so total production is 10,500 pairs.

Try changing `backlog_cost` in the data cell and choose **Run All**. How does a larger penalty change the willingness to delay demand? Even when deliveries are late, they must still be completed by the end of month 4. Save a personal copy in `student-work/` if you want to keep your edits.